# LangChain Retriever Tool Reference

Developer-facing statements defined in `langchain_core.tools.retriever`.

# `RetrieverInput: BaseModel`

Input schema used by retriever tools.

## Field

```python
query: str # Query passed to the retriever
```

---

# `create_retriever_tool`

Converts a `BaseRetriever` into a synchronous and asynchronous `StructuredTool`.

## Syntax

```python
create_retriever_tool(
    retriever: BaseRetriever, # Retriever used to find documents
    name: str, # Tool name exposed to the language model
    description: str, # Tool description exposed to the language model
    *,
    document_prompt: BasePromptTemplate[str] | None = None, # Prompt used to format each document
    document_separator: str = "\n\n", # Separator placed between formatted documents
    response_format: Literal[
        "content",
        "content_and_artifact",
    ] = "content", # Tool output format
) -> StructuredTool # Return the generated retriever tool
```

## Behaviour

- Uses `RetrieverInput` as the generated tool's argument schema.
- Calls `retriever.invoke()` during synchronous execution.
- Calls `retriever.ainvoke()` during asynchronous execution.
- Forwards tool callbacks to the retriever.
- Uses `{page_content}` as the default document prompt.
- Joins formatted documents using `document_separator`.
- Returns formatted text when `response_format="content"`.
- Returns `(formatted_text, documents)` when `response_format="content_and_artifact"`.

In [1]:
from langchain_core.callbacks import CallbackManagerForRetrieverRun # Import the retriever callback manager
from langchain_core.documents import Document # Import the document type
from langchain_core.retrievers import BaseRetriever # Import the abstract retriever base class
from langchain_core.tools import BaseTool # Import the common tool base type
from langchain_core.tools.retriever import create_retriever_tool # Import the retriever-tool factory


class KeywordRetriever(BaseRetriever): # Create a concrete retriever implementation
    documents: list[Document] # Store the documents that can be searched

    def _get_relevant_documents( # Implement the required synchronous retrieval method
        self, # Current retriever instance
        query: str, # Search query received from the tool
        *,
        run_manager: CallbackManagerForRetrieverRun, # Callback manager supplied by LangChain
    ) -> list[Document]: # Return the matching documents
        return [ # Build the list of relevant documents
            document # Return the current matching document
            for document in self.documents # Check every stored document
            if query.lower() in document.page_content.lower() # Match the query without case sensitivity
        ] # Finish creating the result list


documents: list[Document] = [ # Create sample searchable documents
    Document(page_content="Python is a popular programming language."), # Add a Python document
    Document(page_content="LangChain helps developers build LLM applications."), # Add a LangChain document
    Document(page_content="SQL is used to query relational databases."), # Add an SQL document
] # Finish creating the document list

retriever: KeywordRetriever = KeywordRetriever( # Create the custom retriever
    documents=documents, # Supply the searchable documents
) # Finish creating the retriever

retriever_tool: BaseTool = create_retriever_tool( # Convert the retriever into a StructuredTool
    retriever=retriever, # Supply the retriever
    name="search_knowledge", # Set the tool name
    description="Search the stored programming knowledge", # Set the tool description
    document_separator="\n---\n", # Separate multiple retrieved documents
) # Finish creating the retriever tool

result: str = retriever_tool.invoke( # Execute the generated tool
    {
        "query": "python", # Supply the retrieval query
    }
) # Finish invoking the tool

print("Tool name:", retriever_tool.name) # Display the generated tool name

print("Tool arguments:", retriever_tool.args) # Display the query argument schema

print("Retrieved content:") # Display the result heading

print(result) # Display the formatted retrieved documents

Tool name: search_knowledge
Tool arguments: {'query': {'description': 'query to look up in retriever', 'title': 'Query', 'type': 'string'}}
Retrieved content:
Python is a popular programming language.
